In [6]:
%load_ext autoreload
%autoreload 2

In [9]:
import numpy as np
import GDLModules.util as util
import torch.nn.functional as F
import plotly.express as px
import torch.nn as nn
import torch
import GDLModules.A_CAE as A_CAE
import GDLModules.A_AE as A_AE
from datasets import load_dataset
import matplotlib.pyplot as plt
from sklearn.datasets import make_s_curve
from geomloss import SamplesLoss
import copy
from tqdm import tqdm, trange
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
torus_data = util.generate_quarter_torus(10000)
torus_data_test = util.generate_quarter_torus(10000)
util.plot3d([(torus_data, "torus data")], title="Double Torus")

In [130]:
class AE(nn.Module):
    def __init__(self, x_dim, z_dim, hidden_structure):
        super().__init__()
        self.center = nn.Parameter(torch.zeros(x_dim), requires_grad=True)
        self.x_dim = x_dim
        self.z_dim = z_dim
        self.encoder = util.Sum(
            nn.Linear(x_dim, z_dim, bias=False),
            util.MLP(x_dim, z_dim, hidden_structure, activation=nn.Softplus()),
        )
        self.decoder = util.Sum(
            nn.Linear(z_dim, x_dim, bias=False),
            util.MLP(z_dim, x_dim, hidden_structure, activation=nn.Softplus()),
        )




    def encode(self, x):
        z = self.encoder(x-self.center)
        return z

    def decode(self, z):
        x = self.decoder(z)
        return x + self.center


    def encode_decode(self, x):
        z_latent = self.encode(x)
        x_recon = self.decode(z_latent)
        return x_recon

In [131]:
criterion = SamplesLoss(
        loss="sinkhorn", p=2, blur=0.05, scaling=0.8, backend="tensorized", debias=True
    )


def train_ae_sinkhorn(num_epochs: int, x_train, network: AE, device: torch.device, lr = 0.001):
    network.to(device)
    optimizer = torch.optim.Adam(network.parameters(), lr=lr)
    pbar = trange(num_epochs)
    x_train = x_train.to(device)
    x_train_loader = DataLoader(x_train, batch_size=512, shuffle=True)
    for epoch in pbar:

        for batch in x_train_loader:
            # sample 1000 points
            optimizer.zero_grad()
            z_sample = torch.rand(512, network.z_dim).to(device)
            x_decoded = network.decode(z_sample)
            z_ecoded = network.encode(x_decoded)
            loss = criterion(x_decoded, batch) + F.mse_loss(z_ecoded, z_sample)
            pbar.set_postfix({"loss": loss.item()})
            loss.backward()
            optimizer.step()
    return





In [135]:
ae = AE(x_dim=3, z_dim=2, hidden_structure=[64, 64, 64])
ae = ae.to(device)
ae = ae.train()

In [136]:

train_ae_sinkhorn(num_epochs=400, x_train=torch.tensor(torus_data, dtype=torch.float32).to(device), network=ae, device=device, lr=1e-3)

100%|██████████| 400/400 [04:34<00:00,  1.45it/s, loss=0.00718]


In [137]:
torch.save(ae.state_dict(), "ae_quater_torus_sinkhorn2.pt")

In [145]:
z_sample = torch.rand(1000, ae.z_dim).to(device)
x_decoded = ae.decode(z_sample)
util.plot3d([(x_decoded.detach().cpu().numpy(), "decoded torus"), (torus_data[0:1000], "torus data")], title="Decoded Torus from Latent Space")

# Single Chart Geodesics

In [178]:
point_a = np.array([0.1, 0.1])
point_b = np.array([0.9, 0.9])

#linear interpolation
num_points = 10
interpolated_points = torch.tensor(np.linspace(point_a, point_b, num=num_points), device=device, dtype=torch.float)

len(interpolated_points)

decoded_points = ae.decode(torch.tensor(interpolated_points, dtype=torch.float32).to(device)).detach().cpu().numpy()

C:\Users\anton\AppData\Local\Temp\ipykernel_18972\1038090147.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  decoded_points = ae.decode(torch.tensor(interpolated_points, dtype=torch.float32).to(device)).detach().cpu().numpy()


In [159]:
z_sample = torch.rand(1000, ae.z_dim).to(device)
x_decoded = ae.decode(z_sample)
util.plot3d([(x_decoded.detach().cpu().numpy(), "decoded torus"), (decoded_points, "interpolated_points", "lines", 10)], title="Decoded Torus from Latent Space")

In [198]:
z_dim = 2
lr = 0.01
geodesic = interpolated_points.clone()
for j in range (100):
    pointwise_grads = torch.zeros(num_points, z_dim, device=device)
    idx = torch.arange(num_points, device=device)
    decoded_geodesic = ae.decode(geodesic)
    J = torch.autograd.functional.jacobian(ae.encode, decoded_geodesic)[idx, :, idx, :].to(device)  # [10, 2, 3]
    for i in range(1, num_points-1):
        pointwise_grads[i, :] = -num_points * J[i,:,:] @ (decoded_geodesic[i+1] - 2*decoded_geodesic[i] + decoded_geodesic[i-1])
    geodesic.device
    geodesic -= lr * pointwise_grads


In [ ]:
decoded_geodesic = ae.decode(geodesic)
util.plot3d([(x_decoded.detach().cpu().numpy(), "decoded torus"), (decoded_points, "interpolated_points", "lines", 10)], title="Decoded Torus from Latent Space")

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], device='cuda:0')